# 27 — Multi-Provider Fan-Out
> Same question, three AI models, one answer — GPT-4o-mini, Mistral, and Llama weigh in simultaneously, then a synthesis pass tells you where they agree, where they split, and what the consolidated call is.

*Run all cells. Swap the input in the final code cell with your own data.*

In [ ]:
%pip install -q openai pydantic python-dotenv
import os
os.environ['OPENROUTER_API_KEY'] = 'your-openrouter-key'  # replace

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from pydantic import BaseModel, Field


# --- Data shapes ---

class StrategicOpinion(BaseModel):
    model: str = Field(description="The model that produced this opinion (e.g. openai/gpt-4o-mini).")
    recommendation: str = Field(
        description="The model's top strategic recommendation in one to two sentences."
    )
    key_risks: list[str] = Field(
        description="Up to three key risks the model identifies."
    )
    key_opportunities: list[str] = Field(
        description="Up to three key opportunities the model identifies."
    )
    confidence: str = Field(
        description="Model's self-assessed confidence: high, medium, or low."
    )


class ModelConsensus(BaseModel):
    topic: str = Field(description="The strategic topic or question posed.")
    opinions: list[StrategicOpinion] = Field(
        description="One opinion per model, in the order they were queried."
    )
    points_of_agreement: list[str] = Field(
        description="Themes where all or most models agree."
    )
    points_of_disagreement: list[str] = Field(
        description="Areas where models diverge significantly."
    )
    synthesised_recommendation: str = Field(
        description="A single consolidated recommendation that draws on the majority view across models."
    )


# --- Model configuration ---

MODELS = [
    "openai/gpt-4o-mini",
    "mistralai/mistral-7b-instruct",
    "meta-llama/llama-3.1-8b-instruct",
]

SYNTHESIS_MODEL = "openai/gpt-4o-mini"

_OPINION_SYSTEM = (
    "You are a strategic advisor. Given a business topic, provide:\n"
    "- A concise recommendation (1-2 sentences)\n"
    "- Up to 3 key risks\n"
    "- Up to 3 key opportunities\n"
    "- Your confidence level: high, medium, or low\n"
    "Be specific and direct. No hedging."
)

_SYNTHESIS_SYSTEM = (
    "You are a senior analyst synthesising multiple strategic opinions into one consensus view. "
    "Identify where the models agree, where they diverge, and produce a single consolidated "
    "recommendation that reflects the majority position. Be specific."
)


# --- Core logic ---

def _query_model(client: OpenAI, model: str, topic: str) -> StrategicOpinion:
    completion = client.beta.chat.completions.parse(
        model=model,
        messages=[
            {"role": "system", "content": _OPINION_SYSTEM},
            {"role": "user", "content": f"Topic: {topic}"},
        ],
        response_format=StrategicOpinion,
    )
    opinion: StrategicOpinion = completion.choices[0].message.parsed
    opinion.model = model
    return opinion


def _synthesise(client: OpenAI, topic: str, opinions: list[StrategicOpinion]) -> ModelConsensus:
    opinions_text = "\n\n".join(
        f"Model: {o.model}\n"
        f"Recommendation: {o.recommendation}\n"
        f"Risks: {', '.join(o.key_risks)}\n"
        f"Opportunities: {', '.join(o.key_opportunities)}\n"
        f"Confidence: {o.confidence}"
        for o in opinions
    )
    completion = client.beta.chat.completions.parse(
        model=SYNTHESIS_MODEL,
        messages=[
            {"role": "system", "content": _SYNTHESIS_SYSTEM},
            {
                "role": "user",
                "content": f"Topic: {topic}\n\nOpinions:\n{opinions_text}",
            },
        ],
        response_format=ModelConsensus,
    )
    consensus: ModelConsensus = completion.choices[0].message.parsed
    consensus.topic = topic
    consensus.opinions = opinions
    return consensus


def run(topic: str, models: list[str] | None = None) -> ModelConsensus:
    """Query multiple models in parallel and synthesise a consensus."""
    selected_models = models or MODELS
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )

    opinions: list[StrategicOpinion] = []
    with ThreadPoolExecutor(max_workers=len(selected_models)) as executor:
        futures = {
            executor.submit(_query_model, client, model, topic): model
            for model in selected_models
        }
        for future in as_completed(futures):
            opinions.append(future.result())

    opinions.sort(key=lambda o: selected_models.index(o.model))
    return _synthesise(client, topic, opinions)


print("Ready.")

## The scenario

A Series A founder is deciding whether to build a vertical AI product — purpose-built for one industry — or stay horizontal and serve a broader market. It's a classic focus-vs-reach trade-off with real stakes at this stage of the business.

We'll send that question to three models at the same time, then read back where they agree, where they split, and what the consolidated recommendation is.

In [ ]:
TOPIC = (
    "Should a Series A AI startup build a vertical product focused on one industry "
    "or maintain a horizontal platform to capture a broader market?"
)

consensus = run(TOPIC)

# --- Individual model opinions ---
print(f"{'=' * 64}")
print(f"TOPIC: {consensus.topic}")
print(f"{'=' * 64}")

print(f"\nModels queried: {len(consensus.opinions)}")

for opinion in consensus.opinions:
    print(f"\n  [{opinion.model}]  confidence: {opinion.confidence.upper()}")
    print(f"  {opinion.recommendation}")
    if opinion.key_risks:
        print(f"  Risks:         {' | '.join(opinion.key_risks)}")
    if opinion.key_opportunities:
        print(f"  Opportunities: {' | '.join(opinion.key_opportunities)}")

# --- Where they agree and differ ---
if consensus.points_of_agreement:
    print(f"\n{'─' * 64}")
    print("WHERE THEY AGREE")
    for point in consensus.points_of_agreement:
        print(f"  + {point}")

if consensus.points_of_disagreement:
    print("\nWHERE THEY SPLIT")
    for point in consensus.points_of_disagreement:
        print(f"  ! {point}")

# --- The consolidated call ---
print(f"\n{'─' * 64}")
print("CONSOLIDATED RECOMMENDATION")
print(f"  {consensus.synthesised_recommendation}")
print(f"{'=' * 64}")

## Use your own data

Replace the `TOPIC` string in the cell above with:
- **Your strategic question** — any go-to-market, product, pricing, or expansion decision
- **Optionally**, swap the `MODELS` list in Cell 2 to use different providers available on OpenRouter

The agent returns each model's recommendation, risks, and opportunities side by side, then tells you exactly where the models align and where they split — so you know which parts of the decision have consensus and which carry genuine uncertainty.